# 18_descriptors_for_weka — descriptor 계산 & WEKA용 CSV 만들기

**한 줄 요약:** 1:1 학습셋의 분자들에 대해 **descriptor(분자의 물리·화학 수치) 217개**를 계산해서,
(1) 사람이 보는 **전체 Excel**과 (2) WEKA 프로그램에 넣을 **정제된 CSV**를 만든다.

**용어 미리보기**
- **descriptor**: 분자량·지용성(logP)·고리 개수처럼 **분자를 숫자로 요약한 값**. (구조 지문 fingerprint와는 다른 표현)
- **potency**: 우리가 맞히려는 정답. `1`=active(저해함), `0`=inactive(저해 안 함).
- **WEKA**: 217개 중 쓸모 있는 descriptor만 골라주는(feature selection) 도구.

**이 노트북의 큰 흐름:** ① 도구 준비 → ② 1:1 데이터 읽기 → ③ descriptor 217개 계산+Excel저장 → ④ WEKA용으로 정제해 CSV저장

> **📌 이 노트북 읽는 법 (처음이면 여기부터)**
> - **셀(cell)** = 코드 한 덩어리. 위에서부터 하나씩 **실행**(`Shift`+`Enter`)한다.
> - 앞 셀에서 만든 **변수**(값에 붙인 이름표)를 뒤 셀에서 계속 쓴다. 그래서 **순서대로** 실행해야 한다.
> - 코드 줄 뒤의 `# ...` 은 **주석**(설명)이며 실행에 영향이 없다.
> - 자주 나오는 것: `=`(오른쪽 값을 왼쪽 이름에 저장), `[ ]`(리스트=순서 있는 목록), `{ }`(딕셔너리=이름표-값 쌍),
>   `for x in 목록:`(목록을 하나씩 꺼내 반복), `def 함수(입력):`(재사용 작업 묶음),
>   **DataFrame(df)** = 엑셀 표처럼 행·열이 있는 데이터(판다스). `df['열이름']`으로 한 열을 고른다.

### 셀 1 — 준비: 폴더 위치 맞추기 + 도구(라이브러리) 불러오기
**라이브러리** = 남이 만들어 둔 기능 묶음. `import`로 가져와서 쓴다.
- `os`: 폴더/파일 다루기 → 여기선 "지금 위치가 프로젝트 최상위인지" 확인하고 아니면 한 칸 위로 이동(그래야 `data/...` 경로가 맞음).
- `numpy`(np): 숫자 계산, `pandas`(pd): 표 데이터, `rdkit`: 분자 다루기.
- 마지막 줄은 RDKit이 쏟아내는 경고 메시지를 꺼서 화면을 깔끔하게 한다.

In [ ]:
# 노트북을 어느 폴더에서 열든 프로젝트 루트에서 실행되도록 이동
import os                                    # 폴더/파일 관련 기능
if not os.path.isdir('data') and os.path.basename(os.getcwd()) in ('notebooks', 'scripts'):
    os.chdir('..')                           # 'data' 폴더가 안 보이면 상위 폴더로 한 칸 이동
print('작업 폴더:', os.getcwd())              # 지금 작업 폴더를 화면에 출력해 확인

import numpy as np                           # 숫자 계산 도구 (별명 np)
import pandas as pd                          # 표(엑셀 같은) 데이터 도구 (별명 pd)
from rdkit import Chem                       # SMILES(분자 문자열) → 분자 객체로 변환
from rdkit.Chem import Descriptors           # 분자 descriptor(물성 수치) 계산 함수 모음
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')               # RDKit 경고 로그 끄기 (대량 처리 시 화면 정리)

### 셀 2 — 1:1 학습셋 파일 읽기
`pd.read_csv(...)`로 CSV 파일을 **표(df)** 로 읽어온다. 이 표에는 분자 SMILES, inchikey, 정답(label), 출처(source) 열이 있다.
`rename`으로 정답 열 이름 `label`을 **`potency`** 로 바꾼다(의미를 분명히 하려고).
`df.shape`는 (행 개수, 열 개수), `value_counts()`는 값별 개수를 세어 준다 → active/inactive가 균형인지 확인.

In [ ]:
# 1:1 학습셋 로드 (active + 실측 inactive + decoy). label(1/0) -> potency
SRC = 'data/train_1to1.csv'                  # 읽어올 파일 경로를 SRC라는 이름에 저장
df = pd.read_csv(SRC)                         # CSV를 표(df)로 읽기
df = df.rename(columns={'label': 'potency'})  # 'label' 열 이름을 'potency'로 변경
print('[0] 1:1 학습셋 shape:', df.shape)      # (행 수, 열 수) 출력
print('    potency 분포:', dict(df.potency.value_counts()))  # 1/0 각각 몇 개인지
print('    source 분포:', dict(df.source.value_counts()))    # active/decoy/real_inactive 개수

### 셀 3 — descriptor 217개 계산 → 전체 Excel 저장
**흐름:** 분자 하나씩 꺼내서(`for` 반복) → SMILES를 분자로 변환 → descriptor 217개를 한 번에 계산 → 값들을 `rows`에 차곡차곡 모음.
- `Descriptors._descList`: RDKit이 제공하는 descriptor **이름 목록**(217개).
- `Chem.MolFromSmiles`: 글자(SMILES)를 실제 분자 구조로 바꿈. 실패하면 `None` → 그 분자는 건너뜀(`continue`).
- 모은 값을 표 `X`로 만들고, 분자 정보(meta)와 좌우로 붙여(`concat`) 최종 표 `full`을 만든 뒤 Excel로 저장.
- 217개 × 수천 분자라 **수 분 걸린다**(1000개마다 진행 표시).

In [ ]:
# RDKit 2D descriptor 217종 계산 -> 전체 Excel 저장(메타 + potency + descriptor)
desc_names = [n for n, _ in Descriptors._descList]   # descriptor 이름 217개를 리스트로
print('descriptor', len(desc_names), '종 계산 중... (수 분 소요)')

rows, keep = [], []                                  # rows=계산값 담을 곳, keep=성공한 행 번호
for i, smi in enumerate(df['canonical_smiles']):     # 분자를 하나씩(i=번호, smi=SMILES) 반복
    m = Chem.MolFromSmiles(str(smi))                 # SMILES → 분자 객체
    if m is None:                                    # 변환 실패하면
        continue                                     #   이 분자는 건너뛰기
    d = Descriptors.CalcMolDescriptors(m)            # descriptor 217개를 한 번에 계산(딕셔너리)
    rows.append([d.get(n, np.nan) for n in desc_names])  # 이름 순서대로 값만 뽑아 한 줄로 추가
    keep.append(i)                                   # 성공한 행 번호 기록
    if (i + 1) % 1000 == 0:                          # 1000개마다
        print('  ', i + 1, '/', len(df))             #   진행 상황 출력

X = pd.DataFrame(rows, columns=desc_names)           # 계산값들을 표로 (열이름=descriptor명)
meta = df.iloc[keep][['canonical_smiles', 'inchikey', 'source', 'potency']].reset_index(drop=True)  # 성공한 행의 분자정보
full = pd.concat([meta, X.reset_index(drop=True)], axis=1)  # 분자정보 + descriptor 를 옆으로 결합

XLSX = 'data/HSD17B13_1to1_descriptors.xlsx'
full.to_excel(XLSX, index=False)                     # 전체 표를 Excel로 저장
print('[Excel] 전체 저장:', XLSX, '| shape', full.shape,
      '(메타 4 + descriptor', len(desc_names), ')')

### 셀 4 — WEKA에 넣을 CSV로 정제
WEKA는 **① 숫자만, ② 정답(potency)은 맨 마지막 열, ③ 글자 열은 제거, ④ 빈 값(결측) 없음** 을 원한다. 그래서:
1. `remove_invalid_descriptors`: 숫자로 못 바꾸는(문자·이상값) 열을 찾아 버린다. (`try/except` = 시도해보고 실패하면 처리)
2. `inf`(무한대)를 `NaN`(빈 값)으로 바꾼 뒤, **빈 값이 있는 행**을 통째로 지운다 → 모든 칸이 숫자로 채워진 깨끗한 표.
3. 정답 `potency`를 **맨 끝 열**에 붙여 CSV로 저장. (실제로 어떤 descriptor를 고를지는 WEKA에서 한다)

In [ ]:
# ===== WEKA용 CSV 정제 =====
# 규칙: 숫자형 descriptor만 + 클래스(potency)는 '마지막 열' + 문자열 컬럼 삭제 + 결측 없음
compound_info = full[['canonical_smiles', 'inchikey', 'source', 'potency']]  # 분자 정보(글자 등)
descriptor_data = full[desc_names]                   # descriptor 열만 따로
print('[1] descriptor 데이터 shape:', descriptor_data.shape)

# 1) 숫자로 변환 안 되는(문자/이상값) 컬럼 제거
def remove_invalid_descriptors(data):                # 함수 정의: 표를 받아 정제된 표 반환
    invalid = []                                     # 버릴 열 이름 모을 리스트
    for col in data.columns:                         # 열을 하나씩 검사
        try:
            pd.to_numeric(data[col], errors='raise') # 숫자로 바꿔보기
        except Exception:                            # 실패하면(=문자/이상값)
            invalid.append(col)                      #   버릴 목록에 추가
    print('[2] 숫자변환 실패(문자/이상) 컬럼 제거:', len(invalid), '개')
    return data.drop(columns=invalid)                # 그 열들을 뺀 표 반환

desc_num = remove_invalid_descriptors(descriptor_data).apply(pd.to_numeric)  # 정제 후 전부 숫자형으로

# 2) inf -> NaN 후, 빈 값이 있는 '행' 제거 (모든 descriptor 열은 유지)
desc_num = desc_num.replace([np.inf, -np.inf], np.nan)  # 무한대 → 빈 값 처리
n_nan_cell = int(desc_num.isna().sum().sum())        # 빈 값 칸의 총 개수
nan_row_mask = desc_num.isna().any(axis=1)           # 빈 값이 하나라도 있는 '행' 표시(True/False)
print('[3] 결측/inf 셀', n_nan_cell, '개 -> 결측 포함 행', int(nan_row_mask.sum()), '개 제거')
desc_num = desc_num[~nan_row_mask].reset_index(drop=True)         # 그 행들을 제외(~는 '아닌 것')
potency = compound_info['potency'][~nan_row_mask.values].reset_index(drop=True)  # 정답도 같은 행만 남김

# 3) 정답(potency)을 '마지막 열'에 붙이고 저장 (글자 열은 애초에 뺐음)
weka_df = desc_num.copy()                             # descriptor 표 복사
weka_df['potency'] = potency.values                  # 맨 끝에 정답 열 추가
print('[4] WEKA용 shape (descriptor + potency):', weka_df.shape)
print('    마지막 열:', weka_df.columns[-1], '| 클래스 분포:', dict(weka_df.potency.value_counts()))

CSV = 'data/HSD17B13_1to1_descriptors_weka.csv'
weka_df.to_csv(CSV, index=False)                     # CSV로 저장
print('[5] WEKA용 CSV 저장 완료:', CSV)
print('    (WEKA Explorer -> Open file -> Select attributes 탭에서 사용)')